# 🧠 NeuroRestore AI: Master Google Colab Pipeline
**Multi-Center Brain MRI Restoration (VAE-GAN) & Stroke Lesion Segmentation (U-Net)**

This single notebook handles the entire pipeline end-to-end:
1. **Setup & Dependencies** (PyTorch, MONAI, NiBabel)
2. **Mount Google Drive** (to save your trained `.pth` weights)
3. **Data Acquisition & Preprocessing** (ISLES 2022 / Synthetic Fallback)
4. **Train VAE-GAN Restoration Model** ➡️ Saves `vaegan_best.pth`
5. **Workflow 1 (Baseline):** Raw / Noisy MRI ➡️ U-Net ➡️ Saves `unet_baseline.pth`
6. **Workflow 2 (Proposed):** VAE-Restored MRI ➡️ U-Net ➡️ Saves `unet_restored.pth`
7. **Scientific Ablation Comparison:** Side-by-side Dice Score & Visual Comparison!


### Step 1: Install Required Libraries


In [ ]:
# Install specialized medical imaging packages
!pip install -q monai nibabel tqdm matplotlib scipy


### Step 2: Mount Google Drive to Save & Load Models


In [ ]:
import os
from google.colab import drive

# Mount drive
drive.mount('/content/drive')

# Directory where trained weights (.pth) will be saved (in your MODELS folder!)
SAVE_DIR = '/content/drive/MyDrive/MODELS'
os.makedirs(SAVE_DIR, exist_ok=True)
print(f"Models and weights will be saved to: {SAVE_DIR}")


### Step 3: Imports & Hardware Check


In [ ]:
import glob
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from tqdm import tqdm

# Set seed for reproducibility
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device} (Make sure Runtime -> Change runtime type -> T4 GPU is selected)")


### Step 4: Dataset Preparation (Using `processed.zip` from Drive)
Automatically extracts your partner's preprocessed dataset (`processed.zip`) from your `MODELS` folder on Google Drive!


In [ ]:
# Data source: 'PROCESSED_ZIP' (from your MODELS folder on Google Drive)
DATA_SOURCE = "PROCESSED_ZIP"

DATA_DIR = '/content/preprocessed_data'
os.makedirs(DATA_DIR, exist_ok=True)

if DATA_SOURCE == "PROCESSED_ZIP":
    import glob
    import os
    import nibabel as nib
    from scipy.ndimage import zoom

    print("Searching for 'processed.zip' in Google Drive...")
    search_paths = [
        '/content/drive/MyDrive/MODELS/processed.zip',
        '/content/drive/MyDrive/processed.zip',
        '/content/drive/MyDrive/*/processed.zip'
    ]
    found_zip = None
    for p in search_paths:
        matches = glob.glob(p)
        if matches:
            found_zip = matches[0]
            break

    if not found_zip:
        raise FileNotFoundError("Could not find 'processed.zip' in Google Drive. Make sure it is inside 'My Drive/MODELS/'!")

    print(f"Found processed.zip: {found_zip}")
    print("Unzipping preprocessed scans...")
    !unzip -q -o "{found_zip}" -d /content/processed_unzipped/

    # Find all normalized patient cases
    dwi_files = sorted(glob.glob('/content/processed_unzipped/**/normalized/*_dwi_normalized.nii.gz', recursive=True))
    if not dwi_files:
        # Fallback search
        dwi_files = sorted(glob.glob('/content/processed_unzipped/**/*dwi*.nii.gz', recursive=True))

    print(f"Found {len(dwi_files)} patient cases in processed.zip! Extracting 2D slices...")

    count = 0
    # Process cases to extract high-yield slices
    for dwi_p in tqdm(dwi_files):
        msk_p = dwi_p.replace('_dwi_normalized.nii.gz', '_mask.nii.gz')
        if not os.path.exists(msk_p):
            # Look for mask in same directory
            base_dir = os.path.dirname(dwi_p)
            masks = glob.glob(f"{base_dir}/*mask*.nii.gz")
            if masks:
                msk_p = masks[0]
            else:
                continue

        dwi_arr = nib.load(dwi_p).get_fdata().astype(np.float32)
        msk_arr = nib.load(msk_p).get_fdata().astype(np.uint8)

        # Ensure normalized in [0, 1]
        p99 = np.percentile(dwi_arr, 99.5)
        if p99 > 0:
            dwi_arr = np.clip(dwi_arr / p99, 0.0, 1.0)

        for z in range(dwi_arr.shape[2]):
            d_slice = dwi_arr[:, :, z]
            m_slice = msk_arr[:, :, z]

            # Filter out empty background slices
            if np.sum(d_slice > 0.05) < 600:
                continue

            # Standardize in-plane size to 256x256
            if d_slice.shape != (256, 256):
                scale = (256 / d_slice.shape[0], 256 / d_slice.shape[1])
                d_slice = zoom(d_slice, scale, order=1)
                m_slice = zoom(m_slice, scale, order=0)

            np.savez_compressed(
                f"{DATA_DIR}/slice_{count:05d}.npz",
                dwi=d_slice.astype(np.float32),
                mask=(m_slice > 0).astype(np.float32)
            )
            count += 1

    print(f"✅ Extracted {count} clean 2D slices ready for training in {DATA_DIR}!")

elif DATA_SOURCE == "DRIVE_DUMMY_ZIP":
    print("Using dummy_data.zip...")
    !unzip -q -o /content/drive/MyDrive/dummy_data.zip -d {DATA_DIR}
    print(f"Loaded dummy slices to {DATA_DIR}")


### Step 5: PyTorch Dataset with Simulated Noisy Degradation


In [ ]:
class StrokeSliceDataset(Dataset):
    def __init__(self, data_dir, add_noise=True):
        self.files = sorted(glob.glob(os.path.join(data_dir, "*.npz")))
        self.add_noise = add_noise

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        data = np.load(self.files[idx])
        
        # Support both 'dwi' (from dummy_data.zip) and 'image' (from preprocessed slices)
        if 'dwi' in data:
            clean_img = data['dwi'].astype(np.float32)
        elif 'image' in data:
            clean_img = data['image'].astype(np.float32)
        else:
            clean_img = data[data.files[0]].astype(np.float32)

        if 'mask' in data:
            mask = data['mask'].astype(np.float32)
        elif 'msk' in data:
            mask = data['msk'].astype(np.float32)
        else:
            mask = np.zeros_like(clean_img, dtype=np.float32)

        # Create simulated noisy/degraded scan
        if self.add_noise:
            noise = np.random.normal(0, 0.08, clean_img.shape)
            noisy_img = np.clip(clean_img + noise, 0.0, 1.0).astype(np.float32)
        else:
            noisy_img = clean_img.copy()

        return {
            'clean': torch.from_numpy(clean_img).unsqueeze(0),
            'noisy': torch.from_numpy(noisy_img).unsqueeze(0),
            'mask': torch.from_numpy(mask).unsqueeze(0)
        }

# Split into Train and Validation
all_files = sorted(glob.glob(os.path.join(DATA_DIR, "*.npz")))
if len(all_files) == 0:
    raise RuntimeError(f"No .npz files found in {DATA_DIR}! Check Step 4.")

dataset_train = StrokeSliceDataset(DATA_DIR, add_noise=True)
dataset_val   = StrokeSliceDataset(DATA_DIR, add_noise=True)

# Safe batch size for small or large datasets
batch_size = min(4, len(dataset_train))
train_loader = DataLoader(dataset_train, batch_size=batch_size, shuffle=True)
val_loader   = DataLoader(dataset_val, batch_size=batch_size, shuffle=False)
print(f"Total dataset: {len(dataset_train)} slices. Dataloader ready with batch_size={batch_size}!")


### Step 6: Define Residual U-VAE Architecture (Image Restoration)
- **Encoder:** Extracts multi-scale brain features with skip connections
- **Latent Bottleneck:** Spatial variational distribution preserving geometry
- **Decoder with Skip Connections:** Reconstructs ultra-sharp scan, preventing lesion blurriness
- **PatchGAN Discriminator:** Enforces realistic high-frequency MRI textures


In [ ]:
class ResBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(channels, channels, 3, padding=1),
            nn.BatchNorm2d(channels),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(channels, channels, 3, padding=1),
            nn.BatchNorm2d(channels)
        )
    def forward(self, x):
        return x + self.conv(x)

class VAE_Generator(nn.Module):
    def __init__(self):
        super().__init__()
        # Encoder
        self.enc1 = nn.Sequential(nn.Conv2d(1, 32, 3, padding=1), nn.BatchNorm2d(32), nn.LeakyReLU(0.2)) # 256x256
        self.down1 = nn.Sequential(nn.Conv2d(32, 64, 4, 2, 1), nn.BatchNorm2d(64), nn.LeakyReLU(0.2))   # 128x128
        self.down2 = nn.Sequential(nn.Conv2d(64, 128, 4, 2, 1), nn.BatchNorm2d(128), nn.LeakyReLU(0.2)) # 64x64
        self.down3 = nn.Sequential(nn.Conv2d(128, 256, 4, 2, 1), nn.BatchNorm2d(256), nn.LeakyReLU(0.2))# 32x32

        # Latent spatial bottleneck
        self.res = ResBlock(256)
        self.conv_mu = nn.Conv2d(256, 32, 1)
        self.conv_logvar = nn.Conv2d(256, 32, 1)
        self.conv_z = nn.Conv2d(32, 256, 1)

        # Decoder with U-Net Skip Connections (preserves stroke sharpness!)
        self.up3 = nn.Sequential(nn.ConvTranspose2d(256, 128, 4, 2, 1), nn.BatchNorm2d(128), nn.ReLU()) # 64x64
        self.dec3 = nn.Sequential(nn.Conv2d(256, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU())
        
        self.up2 = nn.Sequential(nn.ConvTranspose2d(128, 64, 4, 2, 1), nn.BatchNorm2d(64), nn.ReLU())   # 128x128
        self.dec2 = nn.Sequential(nn.Conv2d(128, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU())

        self.up1 = nn.Sequential(nn.ConvTranspose2d(64, 32, 4, 2, 1), nn.BatchNorm2d(32), nn.ReLU())    # 256x256
        self.dec1 = nn.Sequential(nn.Conv2d(64, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU())

        # Residual delta addition (Learns to subtract noise directly)
        self.out_delta = nn.Sequential(nn.Conv2d(32, 1, 3, padding=1), nn.Tanh())

    def encode(self, x):
        e1 = self.enc1(x)
        e2 = self.down1(e1)
        e3 = self.down2(e2)
        e4 = self.down3(e3)
        feat = self.res(e4)
        mu = self.conv_mu(feat)
        logvar = self.conv_logvar(feat)
        return mu, logvar, (e1, e2, e3)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z, skips):
        e1, e2, e3 = skips
        d = self.conv_z(z)
        d3 = self.up3(d)
        d3 = self.dec3(torch.cat([d3, e3], dim=1))
        d2 = self.up2(d3)
        d2 = self.dec2(torch.cat([d2, e2], dim=1))
        d1 = self.up1(d2)
        d1 = self.dec1(torch.cat([d1, e1], dim=1))
        delta = self.out_delta(d1) * 0.5
        return delta

    def forward(self, x):
        mu, logvar, skips = self.encode(x)
        z = self.reparameterize(mu, logvar)
        delta = self.decode(z, skips)
        restored = torch.clamp(x + delta, 0.0, 1.0)
        return restored, mu, logvar

class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 32, 4, 2, 1), nn.LeakyReLU(0.2),
            nn.Conv2d(32, 64, 4, 2, 1), nn.BatchNorm2d(64), nn.LeakyReLU(0.2),
            nn.Conv2d(64, 128, 4, 2, 1), nn.BatchNorm2d(128), nn.LeakyReLU(0.2),
            nn.Conv2d(128, 1, 4, 1, 1), nn.Sigmoid()
        )
    def forward(self, x):
        return self.net(x)

vae = VAE_Generator().to(device)
disc = Discriminator().to(device)
print("Residual U-VAE Restoration Network initialized successfully!")


### Step 7: Train VAE-GAN Restoration Model


In [ ]:
opt_G = torch.optim.AdamW(vae.parameters(), lr=2e-4)
opt_D = torch.optim.AdamW(disc.parameters(), lr=1e-4)

EPOCHS_VAE = 15

print("Training Sharp Residual U-VAE Restoration Model...")
for epoch in range(EPOCHS_VAE):
    vae.train()
    total_l1 = 0
    for batch in train_loader:
        noisy = batch['noisy'].to(device)
        clean = batch['clean'].to(device)

        # 1. Train Discriminator
        restored, mu, logvar = vae(noisy)
        d_real = disc(clean)
        d_fake = disc(restored.detach())
        loss_D = (F.binary_cross_entropy(d_real, torch.ones_like(d_real)) +
                  F.binary_cross_entropy(d_fake, torch.zeros_like(d_fake))) * 0.5
        opt_D.zero_grad()
        loss_D.backward()
        opt_D.step()

        # 2. Train VAE Generator (High-weight L1 + KL Divergence + Adversarial)
        d_fake_for_G = disc(restored)
        loss_adv = F.binary_cross_entropy(d_fake_for_G, torch.ones_like(d_fake_for_G))
        loss_recon = F.l1_loss(restored, clean)
        loss_kl = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp()) / clean.size(0) * 1e-5
        loss_G = 150.0 * loss_recon + loss_kl + 1.0 * loss_adv

        opt_G.zero_grad()
        loss_G.backward()
        opt_G.step()
        total_l1 += loss_recon.item()

    avg_l1 = total_l1 / len(train_loader)
    print(f"Epoch [{epoch+1}/{EPOCHS_VAE}] - Reconstruction L1: {avg_l1:.4f}")

# Save VAE weights
vaegan_path = os.path.join(SAVE_DIR, 'vaegan_best.pth')
torch.save(vae.state_dict(), vaegan_path)
print(f"✅ Saved Sharp VAE-GAN weights to: {vaegan_path}")


### Step 8: Define U-Net Stroke Lesion Segmentation Model


In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_c, out_c, 3, padding=1),
            nn.BatchNorm2d(out_c),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_c, out_c, 3, padding=1),
            nn.BatchNorm2d(out_c),
            nn.ReLU(inplace=True)
        )
    def forward(self, x):
        return self.conv(x)

class UNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.down1 = DoubleConv(1, 32)
        self.down2 = DoubleConv(32, 64)
        self.down3 = DoubleConv(64, 128)
        self.down4 = DoubleConv(128, 256)
        self.pool = nn.MaxPool2d(2)

        self.up3 = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.conv3 = DoubleConv(256, 128)
        self.up2 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.conv2 = DoubleConv(128, 64)
        self.up1 = nn.ConvTranspose2d(64, 32, 2, stride=2)
        self.conv1 = DoubleConv(64, 32)
        self.out = nn.Conv2d(32, 1, 1)

    def forward(self, x):
        c1 = self.down1(x)
        c2 = self.down2(self.pool(c1))
        c3 = self.down3(self.pool(c2))
        c4 = self.down4(self.pool(c3))

        u3 = self.conv3(torch.cat([self.up3(c4), c3], dim=1))
        u2 = self.conv2(torch.cat([self.up2(u3), c2], dim=1))
        u1 = self.conv1(torch.cat([self.up1(u2), c1], dim=1))
        return torch.sigmoid(self.out(u1))

def dice_score(pred, target, smooth=1e-5):
    pred = (pred > 0.5).float()
    # If both target and prediction are empty, perfect match
    if target.sum() == 0 and pred.sum() == 0:
        return 1.0
    intersection = (pred * target).sum()
    return float((2. * intersection + smooth) / (pred.sum() + target.sum() + smooth))

def dice_loss(pred, target):
    smooth = 1e-5
    intersection = (pred * target).sum(dim=(2, 3))
    union = pred.sum(dim=(2, 3)) + target.sum(dim=(2, 3))
    return 1.0 - (2. * intersection + smooth) / (union + smooth)


### Step 9: Workflow 1 (Baseline: Raw / Noisy MRI ➡️ U-Net)
Trains directly on unenhanced noisy MRI scans to establish the baseline performance.


In [ ]:
unet_baseline = UNet().to(device)
opt_base = torch.optim.AdamW(unet_baseline.parameters(), lr=2e-4)

EPOCHS_UNET = 20

print("Training Workflow 1 (Baseline U-Net)...")
for epoch in range(EPOCHS_UNET):
    unet_baseline.train()
    total_loss = 0
    for batch in train_loader:
        x = batch['noisy'].to(device)
        y = batch['mask'].to(device)
        pred = unet_baseline(x)
        loss = dice_loss(pred, y).mean() + F.binary_cross_entropy(pred, y)

        opt_base.zero_grad()
        loss.backward()
        opt_base.step()
        total_loss += loss.item()

    print(f"Baseline Epoch [{epoch+1}/{EPOCHS_UNET}] - Loss: {total_loss/len(train_loader):.4f}")

base_path = os.path.join(SAVE_DIR, 'unet_baseline.pth')
torch.save(unet_baseline.state_dict(), base_path)
print(f"✅ Saved Baseline U-Net weights to: {base_path}")


### Step 10: Workflow 2 (Proposed: VAE-Restored MRI ➡️ U-Net)
Trains the U-Net on scans that have been cleaned and enhanced by our VAE-GAN model.


In [ ]:
unet_restored = UNet().to(device)
opt_restored = torch.optim.AdamW(unet_restored.parameters(), lr=2e-4)

vae.eval()
print("Training Workflow 2 (Proposed Restored U-Net)...")
for epoch in range(EPOCHS_UNET):
    unet_restored.train()
    total_loss = 0
    for batch in train_loader:
        with torch.no_grad():
            x_noisy = batch['noisy'].to(device)
            # Pass through trained VAE to get cleaned/restored scan
            x_enhanced, _, _ = vae(x_noisy)

        y = batch['mask'].to(device)
        pred = unet_restored(x_enhanced)
        loss = dice_loss(pred, y).mean() + F.binary_cross_entropy(pred, y)

        opt_restored.zero_grad()
        loss.backward()
        opt_restored.step()
        total_loss += loss.item()

    print(f"Restored U-Net Epoch [{epoch+1}/{EPOCHS_UNET}] - Loss: {total_loss/len(train_loader):.4f}")

restored_path = os.path.join(SAVE_DIR, 'unet_restored.pth')
torch.save(unet_restored.state_dict(), restored_path)
print(f"✅ Saved Proposed Restored U-Net weights to: {restored_path}")


### Step 11: Scientific Comparison & Ablation Results (Baseline vs. Restored)


In [ ]:
# Evaluate both models on the validation set
unet_baseline.eval()
unet_restored.eval()
vae.eval()

scores_baseline = []
scores_restored = []

with torch.no_grad():
    for batch in val_loader:
        noisy = batch['noisy'].to(device)
        mask = batch['mask'].to(device)

        # Baseline
        p_base = unet_baseline(noisy)
        for i in range(len(noisy)):
            d_base = dice_score(p_base[i:i+1], mask[i:i+1])
            scores_baseline.append(d_base)

        # Restored
        enhanced, _, _ = vae(noisy)
        p_restored = unet_restored(enhanced)
        for i in range(len(noisy)):
            d_restored = dice_score(p_restored[i:i+1], mask[i:i+1])
            scores_restored.append(d_restored)

avg_base = np.mean(scores_baseline)
avg_restored = np.mean(scores_restored)
improvement = ((avg_restored - avg_base) / (avg_base + 1e-6)) * 100

print("="*50)
print("📊 FINAL SCIENTIFIC ABLATION RESULTS")
print("="*50)
print(f"1. Baseline U-Net (Raw/Noisy MRI) Mean Dice:   {avg_base:.4f}")
print(f"2. Proposed VAE-GAN + U-Net Mean Dice:         {avg_restored:.4f}")
print(f"📈 Relative Performance Gain:                  +{improvement:.2f}%")
print("="*50)

# Visual Plotting: Find a slice that has an ACTUAL stroke lesion for clear demonstration
plot_sample = None
for batch in val_loader:
    for idx in range(len(batch['noisy'])):
        if batch['mask'][idx].sum() > 0:
            plot_sample = {
                'noisy': batch['noisy'][idx:idx+1].to(device),
                'clean': batch['clean'][idx:idx+1].to(device),
                'mask': batch['mask'][idx:idx+1].to(device)
            }
            break
    if plot_sample is not None:
        break

if plot_sample is None:
    # fallback to first slice
    plot_sample = {
        'noisy': batch['noisy'][:1].to(device),
        'clean': batch['clean'][:1].to(device),
        'mask': batch['mask'][:1].to(device)
    }

with torch.no_grad():
    s_noisy = plot_sample['noisy']
    s_mask  = plot_sample['mask']
    s_enh, _, _ = vae(s_noisy)
    pred_base = unet_baseline(s_noisy)
    pred_rest = unet_restored(s_enh)

dice_b = dice_score(pred_base, s_mask)
dice_r = dice_score(pred_rest, s_enh if s_mask is None else s_mask)

fig, axes = plt.subplots(1, 5, figsize=(20, 4))
axes[0].imshow(s_noisy[0, 0].cpu(), cmap='gray')
axes[0].set_title("Input (Noisy Scan)")

axes[1].imshow(s_enh[0, 0].cpu(), cmap='gray')
axes[1].set_title("VAE-Restored (Sharp)")

axes[2].imshow(s_mask[0, 0].cpu(), cmap='Reds', interpolation='none')
axes[2].set_title(f"Ground Truth ({int(s_mask.sum().item())} px)")

axes[3].imshow(pred_base[0, 0].cpu() > 0.5, cmap='Blues', interpolation='none')
axes[3].set_title(f"Baseline Pred (Dice: {dice_b:.2f})")

axes[4].imshow(pred_rest[0, 0].cpu() > 0.5, cmap='Greens', interpolation='none')
axes[4].set_title(f"Restored Pred (Dice: {dice_r:.2f})")

for ax in axes: ax.axis('off')
plt.tight_layout()
plt.savefig('/content/ablation_visual_result.png', dpi=150)
plt.show()


### Step 12: Download Weights to Your Laptop for Streamlit!
Run the code below to download the `.pth` files directly to your laptop for the local Streamlit dashboard!


In [ ]:
from google.colab import files

print("Download these files and place them in your local 'weights/' directory:")
files.download(os.path.join(SAVE_DIR, 'vaegan_best.pth'))
files.download(os.path.join(SAVE_DIR, 'unet_baseline.pth'))
files.download(os.path.join(SAVE_DIR, 'unet_restored.pth'))
